# Reproducing Shaer on a Colab A100 (experiments E0 - E3)

Branch `dev` of the fork (`thebazshah/Shaer`), forked from the official repository at commit `bba73cb`.
Everything this notebook produces is written to **your Google Drive** (`shaer-repro-<model>` folder).

Every cell is plain Python that calls a function of [`colab/shaer_colab.py`](https://github.com/thebazshah/Shaer/blob/dev/colab/shaer_colab.py)
(only the package installation in section 3 uses `!` commands). The module runs the repository's scripts in their own isolated Python
environment (next to Colab's own) and streams their output into the cell.

**You never have to wait blindly.** Every long step shows progress bars (downloads, loading, tokenising, generation, scoring, training),
prints a heartbeat line every ~20 s (time running, time since the last output, GPU use), and a step that is both silent and idle
(GPU below 10 % and almost no CPU) for `STALL_MINUTES` is stopped with a message telling you how to resume. Training runs in the
background; `sc.watch_training(cfg)` is a live view with a bar, loss, ETA and GPU, and can restart a hung run from its last checkpoint.

| Step | What | Needs | Roughly |
|---|---|---|---|
| E0 | Re-score the authors' released poems with the repo's evaluator | any runtime (CPU is fine) | minutes |
| E1 | Generate the 3,481 test poems with the released adapter, score them | GPU, 24 GB+ | hours |
| E2 | Same for the untuned base model | GPU, 24 GB+ | hours |
| E3 | Re-train Shaer (QLoRA, 4,236 steps), then generate and score | A100 40/80 GB class | the authors' run took 12.5 h |

**Runtime:** `Runtime > Change runtime type > A100 GPU`. The isolated environment uses the repository's pinned versions
(transformers 4.43.0, peft 0.12.0, bitsandbytes 0.43.3, torch 2.3.1), so Colab's own preinstalled PyTorch / transformers versions do not matter.

**GPU memory: a 40 GB A100 is enough.** Generation needs the 14 GB bf16 model plus a KV cache whose batch size adapts to the free memory
(25-32 prompts per batch on 40 GB). Training (4-bit QLoRA, batch 1, 1,024 tokens, gradient checkpointing on) needs roughly 12-14 GB by
my estimate; step 7a measures and prints the real peak.

**Before you start:** create a Hugging Face **read** token and add it as a Colab secret named `HF_TOKEN` (key icon in the
left sidebar, "Notebook access" on). The paper's base model
[`Navid-AI/Yehia-7B-preview`](https://huggingface.co/Navid-AI/Yehia-7B-preview) is gated; request access with that account.
**If access is not granted**, set `BASE_MODEL = "ALLaM-AI/ALLaM-7B-Instruct-preview"` in the first cell: it is the model
Yehia was fine-tuned from (same Llama architecture and a byte-identical tokenizer, Apache-2.0, not gated). That is a
documented deviation: E1 is skipped (the released adapter was trained on Yehia's weights), and E2/E3 then measure ALLaM, not Yehia.

**If the session disconnects:** run sections 1 to 3 again (they rebuild `/content`), then re-run the step you were in.
Generation resumes from the JSONL on Drive; training resumes from the newest complete checkpoint on Drive (keep `RUN_NAME` unchanged).

## 1. Configuration, Google Drive, secrets

In [22]:
import getpass
from google.colab import drive, userdata

# ---- edit these -------------------------------------------------------------------------------------------
REPO_URL   = "https://github.com/thebazshah/Shaer.git"
BRANCH     = "dev"
# Base model - exactly one of the two supported choices:
#   "Navid-AI/Yehia-7B-preview"            the paper's model (gated: needs the authors' approval)  -> PAPER FLOW
#   "ALLaM-AI/ALLaM-7B-Instruct-preview"   Yehia's public parent (Apache-2.0, not gated)            -> SUBSTITUTE FLOW
BASE_MODEL = "Navid-AI/Yehia-7B-preview"
FORCE_E1 = False       # substitute flow only: run the released adapter on ALLaM anyway (exploratory, not the paper's E1)
RUN_NAME   = "paper_repro_a100"                      # keep identical across restarts => automatic resume
NUM_SAMPLES = 1        # poems per test prompt in E1/E2/E3. The paper reports one; 3-5 lets you also measure best-of-k.
GEN_BATCH_SIZE = "auto"   # or a number. "auto" sizes the batch from free GPU memory (about 7-10 on an L4, 25-32 on a 40 GB A100, 32 on 80 GB)
STALL_MINUTES = 15     # a step that is silent AND idle (GPU < 10 %, almost no CPU) for this long is stopped, with instructions to resume
# -----------------------------------------------------------------------------------------------------------

drive.mount("/content/drive")

def secret(name, prompt=None):
    try:
        value = userdata.get(name)
        if value:
            return value
    except Exception:
        pass
    return getpass.getpass(prompt) if prompt else ""

HF_TOKEN = secret("HF_TOKEN", "Hugging Face READ token (hidden): ")
GITHUB_TOKEN = secret("GITHUB_TOKEN")      # only needed if the fork is private (optional secret)
print("HF token set:", bool(HF_TOKEN))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Hugging Face READ token (hidden): ··········
HF token set: True


## 2. Get the code from GitHub (branch `dev`) and point its output folders at Drive

In [23]:
import importlib, pathlib, subprocess, sys

# bootstrap: fetch the repository, then everything else is a call into colab/shaer_colab.py
REPO = pathlib.Path("/content/Shaer")
git_url = REPO_URL.replace("https://", f"https://x-access-token:{GITHUB_TOKEN}@") if GITHUB_TOKEN else REPO_URL
if (REPO / ".git").exists():
    subprocess.run(["git", "-C", str(REPO), "checkout", "-q", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO), "pull", "-q", "--ff-only", git_url, BRANCH], check=True)
else:
    subprocess.run(["git", "clone", "-q", "--branch", BRANCH, git_url, str(REPO)], check=True)
subprocess.run(["git", "-C", str(REPO), "remote", "set-url", "origin", REPO_URL], check=True)   # keep no token in .git/config

sys.path.insert(0, str(REPO / "colab"))
import shaer_colab as sc
importlib.reload(sc)                       # pick up the latest version if the module was imported before a pull

cfg = sc.make_config(base_model=BASE_MODEL, run_name=RUN_NAME, num_samples=NUM_SAMPLES, gen_batch_size=GEN_BATCH_SIZE,
                     force_e1=FORCE_E1, stall_minutes=STALL_MINUTES, repo_url=REPO_URL, branch=BRANCH, hf_token=HF_TOKEN)
sc.sync_repo(cfg)                          # training checkpoints and all experiment outputs live on Drive

PAPER FLOW: Yehia-7B-preview. E0, E1, E2, E3 run as in the paper.
base model: Navid-AI/Yehia-7B-preview | Drive folder: /content/drive/MyDrive/shaer-repro-yehia | a silent, idle step is stopped after 15 min
code: d9ec50a Change the base model back to Navid-AI because request access is approved | outputs -> /content/drive/MyDrive/shaer-repro-yehia


## 3. Training environment (about 5 minutes on a fresh runtime) and GPU check

The first cell installs into the isolated environment `/content/venv` with the repository's pinned versions
([`requirements-colab.txt`](https://github.com/thebazshah/Shaer/blob/dev/colab/requirements-colab.txt)). It is safe to run again: when
everything is already installed it finishes in seconds, and it repairs an environment left over from an older version of this notebook.
Its output shows the installer's own progress.

The GPU check refuses to continue training on a GPU this stack cannot use faithfully: no bf16 (T4), or a Blackwell
card (compute capability 12.x), for which the pinned torch 2.3.1 / bitsandbytes 0.43.3 have no kernels.

In [24]:
!pip -q install uv
!uv venv --python 3.12 --allow-existing /content/venv
!uv pip install --python /content/venv/bin/python torch==2.3.1 --index-url https://download.pytorch.org/whl/cu121
!uv pip install --python /content/venv/bin/python -r /content/Shaer/colab/requirements-colab.txt
!uv pip check --python /content/venv/bin/python

Using CPython 3.12.13 interpreter at: /usr/bin/python3
Creating virtual environment at: venv
Activate with: source venv/bin/activate
Using Python 3.12.13 environment at: venv
Checked 1 package in 2ms
Using Python 3.12.13 environment at: venv
Checked 19 packages in 6ms
Using Python 3.12.13 environment at: venv
Checked 76 packages in 1ms
All installed packages are compatible


In [25]:
sc.check_environment(cfg)


▶ checking the training environment and the GPU   (expect a few seconds)
torch 2.3.1+cu121 | transformers 4.43.0 | peft 0.12.0 | bitsandbytes 0.43.3 | protobuf 4.25.3
GPU: NVIDIA A100-SXM4-40GB | 39 GB | compute capability 8.0 | bf16 supported: True
E0 (scoring)    : fine on any runtime
E1/E2 (generate): OK
E3 (training)   : OK
++++++++++++++++++++++ DEBUG INFO END ++++++++++++++++++++++
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
Checking that the library is importable and CUDA is callable...
SUCCESS!
Installation was successful!
✔ environment check: done in 00:06


In [26]:
sc.check_hf_access(cfg)       # fails early if the base model or its tokenizer cannot be loaded


▶ checking access to Navid-AI/Yehia-7B-preview and loading its tokenizer   (expect under a minute)
access to Navid-AI/Yehia-7B-preview : OK
slow tokenizer loads OK | vocab 64000 | LlamaTokenizer
✔ access check: done in 00:08


## 4. E0 - re-score the authors' released poems

Uses the repository's own evaluator, unchanged, on the CPU (deterministic). On the 3,481 released poems it reproduces
the paper's table: meter conformity 0.9064, base-meter accuracy 95.17 %, exact count 83.40 %, count adherence 0.9792.

In [27]:
sc.run_e0(cfg)


▶ E0: exporting the authors' released poems and re-scoring them   (expect about 3-5 minutes)
Repo card metadata block was not found. Setting CardData to empty.
wrote 3481 rows -> runs/e0/generations.jsonl

▶ scoring 3,481 poems with the repository's evaluator (CPU, deterministic)   (expect a few minutes)
  … scoring E0  [█░░░░░░░░░░░░░░░░░░░]   7.0%  243/3,481  elapsed 00:10
  … scoring E0  [███░░░░░░░░░░░░░░░░░]  15.9%  552/3,481  elapsed 00:20
  … scoring E0  [█████░░░░░░░░░░░░░░░]  25.0%  870/3,481  elapsed 00:30
  … scoring E0  [███████░░░░░░░░░░░░░]  33.3%  1,160/3,481  elapsed 00:40
  … scoring E0  [████████░░░░░░░░░░░░]  39.9%  1,389/3,481  elapsed 00:50
  … scoring E0  [██████████░░░░░░░░░░]  47.5%  1,654/3,481  elapsed 01:00
  … scoring E0  [███████████░░░░░░░░░]  55.0%  1,914/3,481  elapsed 01:10
  … scoring E0  [█████████████░░░░░░░]  63.0%  2,193/3,481  elapsed 01:20
  … scoring E0  [██████████████░░░░░░]  71.2%  2,478/3,481  elapsed 01:30
  … scoring E0  [████████████████

## 5. E1 - released adapter (`Shaer-AI/Shaer-adapters`, `adapters/fresh_sft/train/best`)

Decoding follows the paper's Table 15 (T 0.55, top-p 0.90, repetition penalty 1.08, 768 new tokens). The base model is
loaded in bf16 without quantisation (the adapter was trained on an NF4 base; the generation script's `--quantize nf4` matches that instead).

In [ ]:
sc.run_generation(cfg, "e1")


▶ E1: generating the test poems with the released Shaer adapter on Yehia-7B-preview   (expect model download + load first (about 5-10 min on a fresh runtime), then the progress bar with an ETA)
[1/5] loading the test prompts ...
[2/5] loading the tokenizer ...
[3/5] loading the base model Navid-AI/Yehia-7B-preview in bfloat16; the first time this downloads about 14 GB ...
  … E1 generation: running 00:30 | silent for 00:20 | GPU 100% busy, 9.7/40.0 GB | CPU 29% | busy
  … E1 generation: running 00:50 | silent for 00:40 | GPU 100% busy, 9.7/40.0 GB | CPU 9% | busy
  … E1 generation: running 01:10 | silent for 01:00 | GPU 100% busy, 9.7/40.0 GB | CPU 9% | busy
  … E1 generation: running 01:30 | silent for 01:20 | GPU 100% busy, 9.7/40.0 GB | CPU 6% | busy
  … E1 generation: running 01:50 | silent for 01:40 | GPU 100% busy, 9.7/40.0 GB | CPU 9% | busy
  … E1 generation: running 02:10 | silent for 02:00 | GPU 100% busy, 9.7/40.0 GB | CPU 4% | busy
  … E1 generation: running 02:47 | silent

## 6. E2 - untuned base model (same prompts, no adapter)

In [ ]:
sc.run_generation(cfg, "e2")

## 7. E3 - re-train

**7a. Smoke test first (about 10 minutes, includes the 15 GB model download).** It trains 6 steps with the real 4-bit
configuration, which proves bitsandbytes, the data path and the checkpointing work on this runtime before you commit hours to it,
and prints the measured peak GPU memory.

In [ ]:
sc.train_smoke_test(cfg)


▶ smoke test: 6 training steps with the real 4-bit setup   (expect about 10-15 min, mostly the 15 GB model download and loading)
2026-11-17 18:38:56,448 | INFO | Starting SFT | mode=sanity run_name=sanity_paper_repro_a100
2026-11-17 18:38:56,450 | INFO | Output dir: /content/drive/MyDrive/shaer-repro-allam/sft_outputs/sanity_check/sanity_paper_repro_a100
2026-11-17 18:38:56,464 | INFO | push_to_hub is off: nothing will be created or uploaded on the Hub (HF_TOKEN is only used to download gated files)
2026-11-17 18:38:56,482 | INFO | Resume decision: {'timestamp_utc': '2026-11-17T18:38:56.469933+00:00', 'mode': 'sanity', 'resume_mode': 'auto', 'result': 'fresh', 'reason': 'auto_local_no_checkpoint', 'checkpoint_repo_path': None, 'local_resume_path': None}
  Generating train split         [░░░░░░░░░░░░░░░░░░░░]   0%  0/109070  elapsed 00:00
  Generating train split         [████░░░░░░░░░░░░░░░░]  18%  20000/109070  elapsed 00:00  ETA 00:00
  Generating train split         [██████░░░░░░░░

**7b. Full run.** Starts in the background and logs to Drive, so closing the browser tab does not stop it (a
disconnected or recycled runtime does; just re-run sections 1-3 and this cell, it resumes from the newest complete
checkpoint, written every 200 steps: at most ~200 steps are lost, and a half-written checkpoint is skipped automatically).
On a 40 GB A100 leave gradient checkpointing on (the default): it needs roughly 12-14 GB. Switching it off
with `gradient_checkpointing=False` is faster and does not change the maths, but I estimate 20-26 GB for it, which is not
verified: try it only if the peak printed by 7a is under about 14 GB, and go back if you see an out-of-memory error.

In [ ]:
sc.start_training(cfg)        # sc.start_training(cfg, gradient_checkpointing=False) for the faster, memory-hungrier variant

▶ training started (pid 27787)
  log: /content/drive/MyDrive/shaer-repro-allam/logs/train_paper_repro_a100.log
  follow it live with: sc.watch_training(cfg)


Follow the run live: a progress bar, loss, ETA and GPU use, refreshed every 20 s. Stopping this cell with the stop button is safe, training
keeps running. If the run gets stuck (no log output for `STALL_MINUTES` and the GPU idle) the view says so; with `auto_restart=True` it
restarts the run from the last checkpoint by itself (up to 3 times), which is useful when you are away. `sc.training_status(cfg)` is the
one-shot version.

In [ ]:
sc.watch_training(cfg)         # sc.watch_training(cfg, auto_restart=True) to let it restart a hung run from the last checkpoint

training: RUNNING | GPU 100% busy, 9.7/40.0 GB
  [░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]   0.8%  step 33/4,236  ETA 13:56:31  (11.94s/it)
  train loss 2.1410 | lr 1.88e-05 | epoch 0.01 | last eval loss 2.3817
  log last updated 00:01 ago
stopped watching; training continues in the background (sc.watch_training(cfg) to look again)


**7c. After training finished:** generate with the best checkpoint (lowest eval loss), score, summarise.

In [ ]:
sc.run_generation(cfg, "e3")

RuntimeError: training has not finished (no run_summary.json yet): run the training steps first

## 8. Results: your runs against the paper (saved to Drive as `results.md`)

In [ ]:
table = sc.results_table(cfg)

In [ ]:
sc.plot_training_curves(cfg)